# SpectraMorph example integration into HyperBench

This notebook brings the full HyperBench workflow together in one place on the SpectraMorph method.

The goal of this notebook is to show how a user could integrate their own real world research models into the HyperBench framework and simply start generating experiments.

In [1]:
# HyperBench imports
from pathlib import Path
import csv

import cv2
import numpy as np

from hyperbench import (
    __version__,
    BenchmarkConfig,
    DegradationSpec,
    PipelineAdapter,
    load_hsi,
    normalize_image,
    print_data_stats,
    run_benchmark,
)

print("HyperBench version:", __version__)

# SpectraMorph imports
import scipy.io as sio
import numpy as np
from tqdm import tqdm
import os
import math
import time
import io as iot
import sys
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF
import tensorflow as tf
from tensorflow.keras.layers import Dense, ReLU, LeakyReLU
from tensorflow.keras.models import Model
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

HyperBench version: 0.1.0


2026-03-31 18:19:31.448318: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-31 18:19:31.469198: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-31 18:19:31.488590: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-31 18:19:31.494442: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-31 18:19:31.508987: I tensorflow/core/platform/cpu_feature_guar

## Scene configuration

This notebook assumes a MATLAB scene is available locally.

In [2]:
SCENE_PATH = Path("../../data/DC_data.mat")
SCENE_KEY = "dc"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Scene path:", SCENE_PATH)
print("Scene key:", SCENE_KEY)

Scene path: ../../data/DC_data.mat
Scene key: dc


## The SpectraMorph implementation code (taken from the author GitHub and lightly adapted to pass metrics to HyperBench instead of printing them)

In [3]:
def numpy_to_tf(np_array):
    """Convert a NumPy array to a TensorFlow tensor."""
    return tf.constant(np_array, dtype=tf.float32)


def tf_to_numpy(tf_tensor):
    """Convert a TensorFlow tensor to a NumPy array."""
    return tf_tensor.numpy()


def apply_srf_tf(hsi, srf):
    """
    Apply an SRF to an HSI in TensorFlow.

    Parameters
    ----------
    hsi : tf.Tensor
        Hyperspectral image of shape (h, w, C)
    srf : tf.Tensor
        Spectral response function of shape (c, C)

    Returns
    -------
    tf.Tensor
        Low-resolution MSI of shape (h, w, c)
    """
    srf_t = tf.transpose(srf)  # (C, c)
    msi = tf.tensordot(hsi, srf_t, axes=[[-1], [0]])
    return msi

def plot_abundance_maps(abundance_maps, num_endmembers, return_format="png"):
    """
    Plots abundance maps in a grid and returns the resulting figure as image bytes.

    Parameters
    ----------
    abundance_maps : tf.Tensor or np.ndarray
        A 3D tensor/array of shape (height, width, num_endmembers) containing the abundance
        values for each endmember (component) at every spatial location.

    num_endmembers : int
        The number of endmembers (abundance maps) to visualize. This should match the
        last dimension of `abundance_maps`.

    return_format : str, optional
        The format of the image to return ('png' or 'jpg'). Default is 'png'.

    Returns
    -------
    image_bytes : bytes
        The rendered figure as image bytes (PNG or JPEG).
        These bytes can later be written to a file, e.g.:
            with open("abundance_maps.png", "wb") as f:
                f.write(image_bytes)
    """

    cols = 5
    rows = math.ceil(num_endmembers / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    axes = axes.flatten()

    # Compute color scale limits
    vmin = tf.reduce_min(abundance_maps).numpy()
    vmax = tf.reduce_max(abundance_maps).numpy()

    # Plot each abundance map
    for i in range(num_endmembers):
        im = axes[i].imshow(abundance_maps[:, :, i], cmap='viridis', vmin=vmin, vmax=vmax)
        axes[i].set_title(f'Abundance Map {i+1}')
        axes[i].axis('off')

    # Hide unused subplots
    for j in range(num_endmembers, len(axes)):
        axes[j].axis('off')

    # Shared colorbar
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
    fig.colorbar(im, cax=cbar_ax, label='Abundance')

    plt.tight_layout(rect=[0, 0, 0.88, 1])

    # Save to in-memory buffer
    buf = iot.BytesIO()
    fig.savefig(buf, format=return_format, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    image_bytes = buf.getvalue()
    buf.close()

    return image_bytes

def plot_endmember_signatures(endmember_signatures,
                              components=None,
                              normalize=False,
                              title=None,
                              figsize=(11, 6),
                              legend_loc="upper left",
                              return_format="png"):
    """
    Plots spectral signatures for each endmember and returns the resulting figure as image bytes.

    Parameters
    ----------
    endmember_signatures : np.ndarray or tf.Tensor
        Shape (K, C), where K = number of endmembers and C = number of spectral bands/components.

    components : array-like or None, optional
        X-axis values (length C). If None, uses 1..C and labels the axis as 'Components'.

    normalize : bool, optional
        If True, min-max normalizes each signature independently to [0, 1] before plotting.

    title : str or None, optional
        Optional plot title for the figure.

    figsize : tuple, optional
        Size of the figure (width, height). Default is (11, 6).

    legend_loc : str, optional
        Location of the legend (e.g., 'upper left', 'lower right').

    return_format : str, optional
        The format of the returned image ('png' or 'jpg'). Default is 'png'.

    Returns
    -------
    image_bytes : bytes
        The rendered plot as image bytes (PNG or JPEG). These can later be saved using:
            with open("endmember_signatures.png", "wb") as f:
                f.write(image_bytes)
    """
    # --- Convert TensorFlow tensor to NumPy if necessary ---
    try:
        E = endmember_signatures.numpy()
    except AttributeError:
        E = np.asarray(endmember_signatures)

    if E.ndim != 2:
        raise ValueError(f"`endmember_signatures` must be 2D (K, C), got {E.shape}.")

    K, C = E.shape

    # --- X-axis setup ---
    if components is None:
        x = np.arange(1, C + 1)
        x_label = "Components"
    else:
        x = np.asarray(components)
        if x.size != C:
            raise ValueError(f"`components` length ({x.size}) must equal C ({C}).")
        x_label = "Components"

    # --- Optional normalization ---
    if normalize:
        e_min = E.min(axis=1, keepdims=True)
        e_max = E.max(axis=1, keepdims=True)
        E = (E - e_min) / np.maximum(e_max - e_min, 1e-12)

    # --- Plot setup ---
    fig, ax = plt.subplots(figsize=figsize)
    for k in range(K):
        ax.plot(x, E[k], linewidth=2, label=f"Endmember {k+1}")

    ax.set_xlabel(x_label)
    ax.set_ylabel("Intensity")
    if title:
        ax.set_title(title)
    ax.grid(True, alpha=0.3)

    # Legend formatting
    leg = ax.legend(loc=legend_loc, frameon=True, fancybox=False, framealpha=1.0)
    leg.get_frame().set_edgecolor("gray")

    plt.tight_layout()

    # --- Save to in-memory buffer (no GUI required) ---
    buf = iot.BytesIO()
    fig.savefig(buf, format=return_format, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    image_bytes = buf.getvalue()
    buf.close()

    return image_bytes
        
def perform_endmember_extraction(hsi, num_endmembers):
    """
    Performs endmember extraction using Non-negative Matrix Factorization (NMF).

    Parameters
    ----------
    hsi : np.ndarray
        Hyperspectral image cube with shape (H, W, C), where:
            H = image height,
            W = image width,
            C = number of spectral bands.

    num_endmembers : int
        The number of endmembers (distinct material signatures) to extract.

    Returns
    -------
    endmember_signatures : np.ndarray
        Array of shape (num_endmembers, C), where each row is an estimated
        endmember spectral signature.

    Notes
    -----
    - The function uses NMF to decompose the hyperspectral data into an abundance
      matrix (non-negative linear mixture coefficients) and a matrix of
      endmember signatures.
    - The NMF model is initialized using the 'nndsvda' strategy for stable results.
    - The abundance maps themselves are not returned here — only the spectral
      signatures of the endmembers.
    """

    # Clip hyperspectral values to ensure they are within [0, 1]
    # This helps stabilize NMF optimization, which assumes non-negative inputs.
    hsi = np.clip(hsi, 0.0, 1.0)

    # Unpack hyperspectral image dimensions
    H, W, c = hsi.shape

    # Reshape the 3D hyperspectral cube (H, W, C) into a 2D matrix (N, C),
    # where each row represents one pixel's spectral vector.
    V = hsi.reshape(-1, c)

    # Initialize NMF model:
    # - n_components = num_endmembers defines the number of spectral signatures to extract.
    # - init='nndsvda' provides a good non-negative initialization.
    # - max_iter controls convergence limit.
    # - random_state ensures reproducibility.
    model = NMF(n_components=num_endmembers, init='nndsvda', max_iter=5000, random_state=42)

    # Fit NMF to the hyperspectral data (V) to obtain:
    # - W (abundance matrix): pixel-level mixture coefficients
    # - H (components_): spectral signatures (endmembers)
    _ = model.fit_transform(V)

    # Extract endmember spectral signatures (shape: num_endmembers × C)
    endmember_signatures = model.components_

    # Return only the endmember spectra; abundances can be computed separately
    return endmember_signatures


def prepare_inputs(hr_msi, lr_hsi, srf, num_endmembers, spec_prior=False, prior_downsample=4):
    """
    Prepares the data for training.

    Parameters:
        hr_msi (np.ndarray): The high spatial resolution multispectral image of shape (H,W,c)
        lr_hsi (np.ndarray): The low spatial resolution hyperspectral image of shape (h,w,C)
        srf (np.ndarray): The spectral response function of the MSI sensor (can be approximated gaussians) of shape (msi_bands, hsi_bands)
        spec_prior (boolean): A flag indicating whether the coarse spectral prior should be used
        prior_downsample (int): The downsampling factor to build the coarse spectral prior (only used if spec_prior=True)

    Returns:
        hr_msi (tf.Tensor): The high spatial resolution multispectral image of shape (H,W,c)
        lr_hsi (tf.Tensor): The low spatial resolution hyperspectral image of shape (h,w,C)
        lr_msi (tf.Tensor): The low spatial resolution multispectral image of shape (h,w,c)
        downsampled_lr_hsi (tf.Tensor): The downsampled low spatial resolution coarse spectral prior of shape (h/prior_downsample, w/prior_downsample, C)
        abundance_maps (tf.Tensor): The abundance maps obtained from lr_hsi through NMF based endmember extraction of shape (h,w,num_endmembers)
        endmember_signatures (tf.Tensor): The endmember signatures obtained from lr_hsi through NMF based endmember extraction of shape (num_endmembers, C)
    """

    if spec_prior:
        downsampled_lr_hsi = downsample_image(lr_hsi, prior_downsample)
        downsampled_lr_hsi = numpy_to_tf(downsampled_lr_hsi)
    else:
        downsampled_lr_hsi = None

    # Obtaining the abundance maps and endmember signatures
    endmember_signatures = perform_endmember_extraction(lr_hsi, num_endmembers)

    # Generating LR MSI input and converting to tensorflow tensors    
    hr_msi = numpy_to_tf(hr_msi)
    lr_hsi = numpy_to_tf(lr_hsi)
    srf = numpy_to_tf(srf)
    lr_msi = apply_srf_tf(lr_hsi, srf)
    endmember_signatures = numpy_to_tf(endmember_signatures)

    return hr_msi, lr_msi, lr_hsi, downsampled_lr_hsi, endmember_signatures


def get_gpu_memory_mb():
    """
    Return current GPU memory usage in MB for GPU:0.

    If no compatible GPU is available, return NaN rather than failing.
    """
    try:
        mem_info = tf.config.experimental.get_memory_info("GPU:0")
        return mem_info["current"] / (1024 ** 2)
    except Exception:
        return float("nan")

def infer_and_analyze_model_performance_tf(model, sample_inputs):
    """
    Analyzes model complexity: FLOPs, parameters, inference time, and GPU memory usage.
    
    Parameters:
    - model (tf.keras.Model): The model to evaluate.
    - sample_inputs (list of tf.Tensor): List of input tensors matching the model's expected input.
    """
    # 1) Convert sample_inputs into a concrete function
    inputs = [x for x in sample_inputs if x is not None]
    input_signature = [tf.TensorSpec(shape=inp.shape, dtype=inp.dtype) for inp in inputs]

    # Properly trace the model using a callable
    @tf.function
    def model_fn_1(msi):
        return model(msi, None)                 # prior defaults to None in model

    @tf.function
    def model_fn_2(msi, prior):
        return model(msi, prior)

    if len(inputs) == 1:
        concrete_func = model_fn_1.get_concrete_function(*input_signature)
        run_fn = model_fn_1
    elif len(inputs) == 2:
        concrete_func = model_fn_2.get_concrete_function(*input_signature)
        run_fn = model_fn_2
    else:
        raise ValueError("sample_inputs must be [msi] or [msi, prior].")

    # 2) Freeze the graph
    frozen_func = convert_variables_to_constants_v2(concrete_func)
    graph_def = frozen_func.graph.as_graph_def()

    # 3) Compute FLOPs
    try:
        original_stdout = sys.stdout
        sys.stdout = iot.StringIO()

        with tf.Graph().as_default() as graph:
            tf.compat.v1.import_graph_def(graph_def, name="")
            run_meta = tf.compat.v1.RunMetadata()
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            opts["output"] = "none"
            flops = tf.compat.v1.profiler.profile(
                graph=graph,
                run_meta=run_meta,
                options=opts
            ).total_float_ops
    finally:
        sys.stdout = original_stdout

    # 4) Count parameters and record starting GPU memory
    num_params = np.sum([np.prod(v.shape) for v in model.trainable_variables])
    start_mem = get_gpu_memory_mb()

    # 5) Time inference
    start = time.perf_counter()
    SR_image, abundances = run_fn(*inputs)
    end = time.perf_counter()
    inference_time = end - start

    # 6) GPU memory
    end_mem = get_gpu_memory_mb()
    mem_used = end_mem - start_mem

    return SR_image, abundances, num_params, flops, mem_used, inference_time

def batched_inference(model, msi, prior=None, batch_size=None, pbar_desc="Inference"):
    """
    Runs model(msi[, prior]) over spatial tiles and stitches the outputs.
    Expects model to return (sr, abundances) Tensors.

    msi   : np.ndarray | tf.Tensor, shape (H, W, C_msi)
    prior : None | np.ndarray | tf.Tensor, shape (H, W, C_prior) aligned with msi
    batch_size : int | None  (tile size); if None runs full image
    """
    # Convert inputs to NumPy for easy slicing
    if isinstance(msi, tf.Tensor):   msi   = msi.numpy()
    if isinstance(prior, tf.Tensor): prior = prior.numpy() if prior is not None else None

    H, W, _ = msi.shape
    bs = batch_size or max(H, W)
    ys = list(range(0, H, bs))
    xs = list(range(0, W, bs))

    SR_full = None
    AB_full = None

    pbar = tqdm(total=len(ys) * len(xs), desc=pbar_desc)
    for y0 in ys:
        y1 = min(H, y0 + bs)
        for x0 in xs:
            x1 = min(W, x0 + bs)

            patch_msi   = tf.constant(msi[y0:y1, x0:x1, :], dtype=tf.float32)
            patch_prior = None if prior is None else tf.constant(prior[y0:y1, x0:x1, :], dtype=tf.float32)

            sr_patch, ab_patch = model(patch_msi, patch_prior)  # -> Tensors
            sr_patch = sr_patch.numpy()
            ab_patch = ab_patch.numpy()

            # Lazy allocate output arrays using channel dims from first patch
            if SR_full is None:
                SR_full = np.zeros((H, W, sr_patch.shape[-1]), dtype=sr_patch.dtype)
            if AB_full is None:
                AB_full = np.zeros((H, W, ab_patch.shape[-1]), dtype=ab_patch.dtype)

            SR_full[y0:y1, x0:x1, :] = sr_patch
            AB_full[y0:y1, x0:x1, :] = ab_patch

            pbar.update(1)
    pbar.close()
    return SR_full, AB_full

class MSItoHSI_MLP(Model):
    def __init__(self, endmember_signatures, hidden_size=1024):
        """
        Initializes a small MLP model that estimates abundances from MSI and reconstructs HSI using fixed endmember signatures.

        Parameters:
            endmember_signatures (np.ndarray or tf.Tensor): Matrix of shape (K, C_hsi)
            hidden_size (int): Number of units in each hidden layer
        """
        super(MSItoHSI_MLP, self).__init__()
        
        # Store fixed endmember signatures E (K, C_hsi)
        E = tf.convert_to_tensor(endmember_signatures, dtype=tf.float32)
        self.E = tf.constant(E, dtype=tf.float32)
        self.K = int(E.shape[0])
        self.C_hsi = int(E.shape[1])
        
        # Define the MLP layers explicitly
        self.layer1 = Dense(hidden_size, activation='relu', dtype=tf.float32)
        
        # Abundance layer (K outputs, softmax for sum-to-one)
        self.abundance_layer = Dense(self.K, activation='linear', dtype=tf.float32)

    def call(self, msi, prior_hsi):
        """
        Forward pass of the model.

        Parameters:
            msi (tf.Tensor): MSI image of shape (H, W, c_msi)
            prior_hsi (tf.Tensor): Downsampled lr hsi to construct Coarse Spectral Prior for training from

        Returns:
            hsi_hat (tf.Tensor): Reconstructed HSI image of shape (H, W, C_hsi)
        """
        # 1) Grab dynamic sizes
        H = tf.shape(msi)[0]
        W = tf.shape(msi)[1]

        # 2) Flatten inputs
        msi = tf.reshape(msi, [H * W, -1])

        if prior_hsi is not None:
            prior_hsi = tf.reshape(prior_hsi, [H * W, -1])
            x = tf.concat([msi, prior_hsi], axis=-1) # [batch, c+C]
        else:
            x = msi

        # 3) Forward through MLP
        x = self.layer1(x)

        # 4) Estimate abundances
        abundances = self.abundance_layer(x)  # (H*W, K)

        # 5) Linear mixing using endmember signatures E
        hsi_hat_flat = tf.matmul(abundances, self.E)  # (H*W, C_hsi)

        # 6) Reshape outputs
        hsi_hat = tf.reshape(hsi_hat_flat, [H, W, self.C_hsi])     # (H, W, C_hsi)
        abundances = tf.reshape(abundances, [H, W, self.K])        # (H, W, K)

        return hsi_hat, abundances

def train_mlp(
    lr_msi, 
    lr_hsi, 
    lr_lr_hsi, 
    endmember_signatures, 
    epochs, 
    init_lr, 
    max_lr, 
    final_lr, 
    hidden_size,
    batch_size=1024):
    """
    Trains the MLP to estimate abundance maps from msi inputs per pixel.

    Parameters:
        lr_msi (tf.Tensor): Low-res MSI (h,w,c)
        lr_hsi (tf.Tensor): Low-res HSI (h,w,C)
        lr_lr_hsi (tf.Tensor): Downsampled Low res HSI (h/prior_downsample, w/prior_downsample, C) if using Coarse Spectral Prior else None
        endmember_signatures (tf.Tensor): Endmember signatures obtained from NMF (c,C)
        epochs (int): Total number of epochs
        init_lr (float): Initial learning rate
        max_lr (float): Peak learning rate
        final_lr (float): Learning rate at end of training
        hidden_size (int): Number of hidden units in MLP
        batch_size (int): Spatial tile size (B), if None, trains on the full image

    Returns:
        trained_model (tf.keras.Model): Trained MLP model to estimate abundance maps from msi
    """

    # Instantiating the model
    model = MSItoHSI_MLP(endmember_signatures=endmember_signatures, hidden_size=hidden_size)

    H, W, c = lr_msi.shape
    _, _, C = lr_hsi.shape
    
    # Compute prior if given
    if lr_lr_hsi is not None:
        ratio = H // lr_lr_hsi.shape[0]
        x, y = tf.range(W), tf.range(H)
        xx, yy = tf.meshgrid(x, y)
        a = tf.clip_by_value(tf.math.floordiv(yy, ratio), 0, tf.shape(lr_lr_hsi)[0] - 1)
        b = tf.clip_by_value(tf.math.floordiv(xx, ratio), 0, tf.shape(lr_lr_hsi)[1] - 1)
        coords = tf.stack([a, b], axis=-1)
        prior_hsi = tf.gather_nd(lr_lr_hsi, coords)
    else:
        prior_hsi = None

    # Defining the optimizer
    optimizer = tf.keras.optimizers.Adam(learning_rate=init_lr)
    # Defining the loss function
    mae = tf.keras.losses.MeanAbsoluteError()

    # Define One-Cycle LR schedule
    def one_cycle_lr(epoch, total_epochs, init_lr, max_lr, final_lr, pct_up=0.3):
        if epoch < pct_up * total_epochs:
            # Linear ramp-up
            return init_lr + (max_lr - init_lr) * (epoch / (pct_up * total_epochs))
        else:
            # Linear ramp-down
            return max_lr - (max_lr - final_lr) * ((epoch - pct_up * total_epochs) / ((1 - pct_up) * total_epochs))

    # Training loop
    pbar = tqdm(range(1, epochs + 1), desc="Training Model", unit="epoch")
    for epoch in pbar:
        current_lr = one_cycle_lr(epoch, epochs, init_lr, max_lr, final_lr)
        optimizer.learning_rate.assign(current_lr)

        # full-image training
        if batch_size is None:
            with tf.GradientTape() as tape:
                pred_lr_hsi, estimated_abundances = model(lr_msi, prior_hsi)
                loss = mae(lr_hsi, pred_lr_hsi)
            grads = tape.gradient(loss, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            pbar.set_postfix(loss=f"{loss.numpy():.4f}", lr=f"{current_lr:.2e}")

        # tiled/batched training
        else:
            epoch_loss = 0.0
            count = 0

            # Iterate over non-overlapping spatial tiles (ragged edges handled by slicing)
            for y0 in range(0, H, batch_size):
                y1 = min(H, y0 + batch_size)
                for x0 in range(0, W, batch_size):
                    x1 = min(W, x0 + batch_size)

                    sub_msi = lr_msi[y0:y1, x0:x1, :]
                    sub_hsi = lr_hsi[y0:y1, x0:x1, :]

                    # Slice the precomputed prior for the current tile (if available)
                    sub_prior = None if prior_hsi is None else prior_hsi[y0:y1, x0:x1, :]

                    with tf.GradientTape() as tape:
                        pred_lr_hsi, estimated_abundances = model(sub_msi, sub_prior)
                        loss = mae(sub_hsi, pred_lr_hsi)

                    grads = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(grads, model.trainable_variables))

                    epoch_loss += float(loss.numpy())
                    count += 1

            # report average patch loss
            avg_loss = epoch_loss / max(count, 1)
            pbar.set_postfix(average_loss=f"{avg_loss:.4f}", lr=f"{current_lr:.2e}")

    return model

def run_pipeline(
    HR_MSI,
    LR_HSI,
    srf,
    psf=None, 
    metadata=None,
    num_endmembers=7,
    epochs=2500,
    init_lr=1e-3,
    max_lr=1e-2,
    final_lr=1e-6,
    hidden_size=1024,
    spec_prior=False, 
    prior_downsample=4,
    training_batch_size=None,
    inference_batch_size=None,
    synthetic=False,
    return_format='png'
):

    """
    Runs the entire SpectraMorph pipeline end to end.
    """

    # Start training timing
    start_time = time.perf_counter()

    # Obtaining the tensorflow tensors of all the required inputs
    hr_msi, lr_msi, lr_hsi, lr_lr_hsi, endmember_signatures = prepare_inputs(
        HR_MSI, LR_HSI, srf, num_endmembers, spec_prior=spec_prior, prior_downsample=prior_downsample
    )

    # Train the MLP on low resolution input-target pair to learn to estimate abundances from msi per pixel
    trained_model = train_mlp(
        lr_msi, lr_hsi, lr_lr_hsi, endmember_signatures, epochs, init_lr, max_lr, final_lr,
        hidden_size, batch_size=training_batch_size
    )

    # End timing
    end_time = time.perf_counter()
    training_time = end_time - start_time

    H, W, c = hr_msi.shape
    _, _, C = lr_hsi.shape

    if spec_prior:
        h, w, _ = lr_hsi.shape
        ratio = H // h
        x, y = tf.range(W), tf.range(H)
        xx, yy = tf.meshgrid(x, y)
        a = tf.clip_by_value(tf.math.floordiv(yy, ratio), 0, tf.shape(lr_hsi)[0] - 1)
        b = tf.clip_by_value(tf.math.floordiv(xx, ratio), 0, tf.shape(lr_hsi)[1] - 1)
        coords = tf.stack([a, b], axis=-1)
        prior = tf.gather_nd(lr_hsi, coords)
    else:
        prior = None

    if inference_batch_size is None:
        print("Inferring on all input pixels...")
        SR_image, abundances, num_params, flops, mem_used, inference_time = infer_and_analyze_model_performance_tf(
            trained_model,
            sample_inputs=[hr_msi, prior]
        )
        SR_image = tf_to_numpy(SR_image)
    else:
        print("Inferring on ", inference_batch_size*inference_batch_size, " pixels per batch...")
        SR_image, abundances = batched_inference(
            trained_model, hr_msi, prior=prior, batch_size=inference_batch_size
        )

    # We comment the author's endmember signature and abundance like latent estimates plotting
    # endmember_signatures_img = plot_endmember_signatures(endmember_signatures, normalize=False, return_format=return_format)
    # alle_img = plot_abundance_maps(abundances, num_endmembers=num_endmembers, return_format=return_format)

    # Ouputting the super resolved image (HR HSI)
    SR_image = np.clip(SR_image, 0, 1)

    stats = {
        "framework": "tensorflow",
        "training_time_sec": float(training_time),
        "inference_time_sec": float(inference_time),
        "num_parameters": int(num_params),
        "flops": float(flops) if not isinstance(flops, (int, np.integer)) else int(flops),
        "gpu_memory_mb": float(mem_used),
    }
    
    return SR_image, stats

class SpectraMorphPipeline:
    def run_pipeline(self, HR_MSI, LR_HSI, srf, psf=None, metadata=None):
        return run_pipeline(HR_MSI, LR_HSI, srf, psf, metadata)

## Wrap the method with `PipelineAdapter`

`PipelineAdapter` is the standard integration path for the `run_pipeline(...)` interface.

In [4]:
adapter = PipelineAdapter(
        pipeline=SpectraMorphPipeline(),
        name="SpectraMorph",
        input_backend="numpy",
        output_backend="numpy",
        add_batch_dim=False,
        device="auto",
    )

print("Adapter ready:", adapter.name)


Adapter ready: SpectraMorph


## Part 1: Explicit benchmark design with `degradation_specs`

This approach is useful when the experiment set is known in advance and should be controlled exactly.

DegradationSpec allows a user to control the exact downsampling_ratio, msi_band_count, and the Signal to Noise ratios for noise injection into the lr_hsi and hr_msi. A user can specify the exact PSFs that they wish to run these degradations with when they build a benchmark configuration (shown below).

In [5]:

explicit_specs = [
    DegradationSpec(downsample_ratio=4,  msi_band_count=4,  spatial_snr_db=35.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=4,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=16, msi_band_count=4,  spatial_snr_db=25.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=32, msi_band_count=4,  spatial_snr_db=20.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=3,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=8,  spatial_snr_db=30.0, spectral_snr_db=40.0),
    DegradationSpec(downsample_ratio=8,  msi_band_count=16, spatial_snr_db=30.0, spectral_snr_db=40.0),
]

for spec in explicit_specs:
    print(spec)

DegradationSpec(downsample_ratio=4, msi_band_count=4, spatial_snr_db=35.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=4, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=16, msi_band_count=4, spatial_snr_db=25.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=32, msi_band_count=4, spatial_snr_db=20.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=3, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=8, spatial_snr_db=30.0, spectral_snr_db=40.0)
DegradationSpec(downsample_ratio=8, msi_band_count=16, spatial_snr_db=30.0, spectral_snr_db=40.0)


## Build an explicit `BenchmarkConfig`

This configuration uses:

- ten PSFs
- one sigma
- one kernel radius
- exact `degradation_specs`
- clipping disabled
- CSV writing enabled
- per-case flushing enabled

In [6]:

explicit_csv_path = OUTPUT_DIR / "SpectraMorph_jupyter_exact_degradations.csv"

explicit_config = BenchmarkConfig(
    scene_path=SCENE_PATH,
    scene_key=SCENE_KEY,
    psf_names=["gaussian"], # Users can add any of the supported PSFs here
    psf_sigmas=[3.4],
    psf_kernel_radii=[7],
    degradation_specs=explicit_specs,
    clip_prediction_to_unit_interval=False,
    output_csv_path=explicit_csv_path,
    save_csv=True,
    flush_csv_after_each_case=True,
    overwrite_csv_on_start=True,
    fail_fast=True,
)

## Run the explicit benchmark

In [7]:
explicit_results = run_benchmark(adapter, explicit_config)

print("Number of explicit result rows:", len(explicit_results.rows))
print("CSV written to:", explicit_csv_path.resolve())

[2026-03-31 18:19:39] INFO | hyperbench | Loading scene from '../../data/DC_data.mat' with key 'dc'
[2026-03-31 18:19:40] INFO | hyperbench | Generating benchmark cases from config.
[2026-03-31 18:19:40] INFO | hyperbench | Initialized benchmark CSV at 'outputs/SpectraMorph_jupyter_exact_degradations.csv'.
[2026-03-31 18:19:40] INFO | hyperbench | Starting benchmark run with 7 case(s) using adapter 'SpectraMorph'.
[2026-03-31 18:19:40] INFO | hyperbench | Starting case_00000 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=4, snr_spatial=35.0, snr_spectral=40.0, seed=42
2026-03-31 18:20:59.230247: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:60:00.0, compute capability: 7.5
Training Model: 100%|██████████| 2500/2500 [00:59<00:00, 42.17epoch/s, loss=0.0051, lr=1.00e-06]
I0000 00:00:1774981318.820971 1140548 devices.cc:67] Number of eli

Inferring on all input pixels...
Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


[2026-03-31 18:24:14] INFO | hyperbench | Completed case_00000 successfully in 273.728s | RMSE=0.012466, PSNR=38.005189, SSIM=0.990778, UIQI=0.975995, ERGAS=2.813318, SAM=1.934869
[2026-03-31 18:24:14] INFO | hyperbench | Starting case_00001 | psf=gaussian, sigma=3.4, kernel=7, r=8, c=4, snr_spatial=30.0, snr_spectral=40.0, seed=42
Training Model: 100%|██████████| 2500/2500 [01:02<00:00, 39.94epoch/s, loss=0.0071, lr=1.00e-06]
I0000 00:00:1774981565.869333 1140548 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 18:26:05.869474: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 18:26:05.874518: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:60:00.0, compute capability: 7.5


Inferring on all input pixels...


[2026-03-31 18:28:14] INFO | hyperbench | Completed case_00001 successfully in 240.739s | RMSE=0.012868, PSNR=37.734383, SSIM=0.990695, UIQI=0.977070, ERGAS=2.411472, SAM=2.025077
[2026-03-31 18:28:14] INFO | hyperbench | Starting case_00002 | psf=gaussian, sigma=3.4, kernel=7, r=16, c=4, snr_spatial=25.0, snr_spectral=40.0, seed=42
Training Model: 100%|██████████| 2500/2500 [01:02<00:00, 40.09epoch/s, loss=0.0110, lr=1.00e-06]
I0000 00:00:1774981803.664213 1140548 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 18:30:03.664350: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 18:30:03.669401: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:60:00.0, compute capability: 7.5


...

Inferring on all input pixels...


[2026-03-31 18:44:01] INFO | hyperbench | Completed case_00005 successfully in 235.428s | RMSE=0.008977, PSNR=40.793554, SSIM=0.993314, UIQI=0.976178, ERGAS=2.777326, SAM=1.459798
[2026-03-31 18:44:01] INFO | hyperbench | Starting case_00006 | psf=gaussian, sigma=3.4, kernel=7, r=8, c=16, snr_spatial=30.0, snr_spectral=40.0, seed=42
Training Model: 100%|██████████| 2500/2500 [01:01<00:00, 40.48epoch/s, loss=0.0067, lr=1.00e-06]
I0000 00:00:1774982751.554978 1140548 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 18:45:51.555109: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 18:45:51.560285: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:60:00.0, compute capability: 7.5


Inferring on all input pixels...


[2026-03-31 18:47:56] INFO | hyperbench | Completed case_00006 successfully in 235.169s | RMSE=0.009582, PSNR=40.241237, SSIM=0.993745, UIQI=0.977737, ERGAS=2.697261, SAM=1.505989
[2026-03-31 18:47:56] INFO | hyperbench | Benchmark results were incrementally written to 'outputs/SpectraMorph_jupyter_exact_degradations.csv'.
[2026-03-31 18:47:56] INFO | hyperbench | Benchmark run complete. total=7 success=7 failed=0


Number of explicit result rows: 7
CSV written to: /work/pi_mduarte_umass_edu/Ritik_Shah_Folder/HyperBench/examples/SpectraMorph/outputs/SpectraMorph_jupyter_exact_degradations.csv


## Inspect the structured benchmark rows

Each row is a dictionary containing:

- scene and method identifiers
- degradation parameters
- input shapes
- metrics
- status information
- runtime
- optional stats returned by the pipeline

In [8]:
for i, row in enumerate(explicit_results.rows):
    print(f"Row {i}")
    for key, value in row.items():
        print(f"  {key}: {value}")
    print("-" * 100)

Row 0
  scene_name: DC_data
  shape_policy: strict
  method_name: SpectraMorph
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 4
  msi_band_count: 4
  spatial_snr: 35.0
  spectral_snr: 40.0
  fwhm_factor: 4.2
  seed: 42
  gt_shape: (1280, 307, 191)
  lr_hsi_shape: (320, 76, 191)
  hr_msi_shape: (1280, 307, 4)
  prediction_clipped: False
  prediction_clip_min: 0.0
  prediction_clip_max: 1.0
  framework: tensorflow
  training_time_sec: 91.96651948010549
  inference_time_sec: 0.01873679319396615
  num_parameters: 12295
  flops: 10308519681
  gpu_memory_mb: 302.0185546875
  RMSE: 0.012466054349065597
  PSNR: 38.00518885189997
  SSIM: 0.9907783555322065
  UIQI: 0.975995045336706
  ERGAS: 2.8133176568919724
  SAM: 1.9348686933517456
  status: success
  error: 
  runtime_seconds: 273.7284009717405
----------------------------------------------------------------------------------------------------
Row 1
  scene_name: DC_data
  shape_policy: strict
  method_name: Spect

## Part 2: Sweep-style benchmark design

A sweep-style configuration is useful when you want HyperBench to build the experiment set automatically from parameter lists.

This style is appropriate when the benchmark should cover a broader grid of conditions. This example keeps the sweep deliberately simple since sweep style benchmarking generates too many experiments.

In [9]:

sweep_csv_path = OUTPUT_DIR / "SpectraMorph_jupyter_sweep_results.csv"

sweep_config = BenchmarkConfig(
    scene_path=SCENE_PATH,
    scene_key=SCENE_KEY,
    psf_names=["gaussian", "moffat"],
    psf_sigmas=[3.4],
    psf_kernel_radii=[7],
    msi_band_counts=[3, 4, 8],
    downsample_ratio_to_spatial_snr_db={
        4: 35.0,
        8: 30.0,
    },
    spectral_snr_dbs=[40.0],
    clip_prediction_to_unit_interval=True,
    prediction_clip_min=0.0,
    prediction_clip_max=1.0,
    output_csv_path=sweep_csv_path,
    save_csv=True,
    flush_csv_after_each_case=True,
    overwrite_csv_on_start=True,
    fail_fast=True,
)

print(sweep_config)

BenchmarkConfig(scene_path=PosixPath('../../data/DC_data.mat'), scene_key='dc', psf_names=['gaussian', 'moffat'], psf_sigmas=[3.4], psf_kernel_radii=[7], degradation_specs=None, msi_band_counts=[3, 4, 8], downsample_ratio_to_spatial_snr_db={4: 35.0, 8: 30.0}, spectral_snr_dbs=[40.0], fwhm_factor=4.2, seed=42, metrics=['rmse', 'psnr', 'ssim', 'uiqi', 'ergas', 'sam'], normalize_inputs=True, lower_percentile=1.0, upper_percentile=99.0, clip_prediction_to_unit_interval=True, prediction_clip_min=0.0, prediction_clip_max=1.0, save_csv=True, output_csv_path=PosixPath('outputs/SpectraMorph_jupyter_sweep_results.csv'), flush_csv_after_each_case=True, overwrite_csv_on_start=True, fail_fast=True, user_srf=None, user_psf=None, metadata={})


## Run the sweep benchmark

In [10]:
sweep_results = run_benchmark(adapter, sweep_config)

print("Number of sweep result rows:", len(sweep_results.rows))
print("CSV written to:", sweep_csv_path.resolve())

[2026-03-31 18:47:57] INFO | hyperbench | Loading scene from '../../data/DC_data.mat' with key 'dc'
[2026-03-31 18:47:58] INFO | hyperbench | Generating benchmark cases from config.
[2026-03-31 18:47:58] INFO | hyperbench | Initialized benchmark CSV at 'outputs/SpectraMorph_jupyter_sweep_results.csv'.
[2026-03-31 18:47:58] INFO | hyperbench | Starting benchmark run with 12 case(s) using adapter 'SpectraMorph'.
[2026-03-31 18:47:58] INFO | hyperbench | Starting case_00000 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=3, snr_spatial=35.0, snr_spectral=40.0, seed=42
Training Model: 100%|██████████| 2500/2500 [01:05<00:00, 38.44epoch/s, loss=0.0056, lr=1.00e-06]
I0000 00:00:1774983006.051956 1140548 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 18:50:06.052098: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 18:50:06.057150: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /j

Inferring on all input pixels...


[2026-03-31 18:52:11] INFO | hyperbench | Completed case_00000 successfully in 253.419s | RMSE=0.015522, PSNR=36.125684, SSIM=0.987879, UIQI=0.975062, ERGAS=2.841057, SAM=2.194923
[2026-03-31 18:52:11] INFO | hyperbench | Starting case_00001 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=4, snr_spatial=35.0, snr_spectral=40.0, seed=42
Training Model: 100%|██████████| 2500/2500 [01:05<00:00, 38.24epoch/s, loss=0.0052, lr=1.00e-06]
I0000 00:00:1774983260.486156 1140548 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 18:54:20.486297: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 18:54:20.491306: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:60:00.0, compute capability: 7.5


Inferring on all input pixels...


[2026-03-31 18:56:25] INFO | hyperbench | Completed case_00001 successfully in 254.221s | RMSE=0.012547, PSNR=37.949576, SSIM=0.990606, UIQI=0.975055, ERGAS=2.784560, SAM=1.946906
[2026-03-31 18:56:25] INFO | hyperbench | Starting case_00002 | psf=gaussian, sigma=3.4, kernel=7, r=4, c=8, snr_spatial=35.0, snr_spectral=40.0, seed=42
Training Model: 100%|██████████| 2500/2500 [01:01<00:00, 40.50epoch/s, loss=0.0044, lr=1.00e-06]
I0000 00:00:1774983510.961688 1140548 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 18:58:30.961842: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 18:58:30.966952: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:60:00.0, compute capability: 7.5


...

Inferring on all input pixels...


[2026-03-31 19:33:14] INFO | hyperbench | Completed case_00010 successfully in 241.520s | RMSE=0.012333, PSNR=38.097214, SSIM=0.991084, UIQI=0.976281, ERGAS=2.787543, SAM=1.921496
[2026-03-31 19:33:14] INFO | hyperbench | Starting case_00011 | psf=moffat, sigma=3.4, kernel=7, r=8, c=8, snr_spatial=30.0, snr_spectral=40.0, seed=42
Training Model: 100%|██████████| 2500/2500 [01:07<00:00, 37.12epoch/s, loss=0.0066, lr=1.00e-06]
I0000 00:00:1774985713.764219 1140548 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-03-31 19:35:13.764342: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
2026-03-31 19:35:13.769489: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9611 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:60:00.0, compute capability: 7.5


Inferring on all input pixels...


[2026-03-31 19:37:19] INFO | hyperbench | Completed case_00011 successfully in 244.806s | RMSE=0.008653, PSNR=41.103107, SSIM=0.993390, UIQI=0.975395, ERGAS=3.102369, SAM=1.413993
[2026-03-31 19:37:19] INFO | hyperbench | Benchmark results were incrementally written to 'outputs/SpectraMorph_jupyter_sweep_results.csv'.
[2026-03-31 19:37:19] INFO | hyperbench | Benchmark run complete. total=12 success=12 failed=0


Number of sweep result rows: 12
CSV written to: /work/pi_mduarte_umass_edu/Ritik_Shah_Folder/HyperBench/examples/SpectraMorph/outputs/SpectraMorph_jupyter_sweep_results.csv


## Inspect the sweep rows

In [11]:
for i, row in enumerate(sweep_results.rows):
    print(f"Sweep row {i}")
    for key, value in row.items():
        print(f"  {key}: {value}")
    print("-" * 100)

Sweep row 0
  scene_name: DC_data
  shape_policy: strict
  method_name: SpectraMorph
  psf_name: gaussian
  sigma: 3.4
  kernel_size: 15
  downsampling_ratio: 4
  msi_band_count: 3
  spatial_snr: 35.0
  spectral_snr: 40.0
  fwhm_factor: 4.2
  seed: 42
  gt_shape: (1280, 307, 191)
  lr_hsi_shape: (320, 76, 191)
  hr_msi_shape: (1280, 307, 3)
  prediction_clipped: True
  prediction_clip_min: 0.0
  prediction_clip_max: 1.0
  framework: tensorflow
  training_time_sec: 82.329843506217
  inference_time_sec: 0.015290683135390282
  num_parameters: 11271
  flops: 9503737601
  gpu_memory_mb: 301.369140625
  RMSE: 0.015521790246217323
  PSNR: 36.125683588736095
  SSIM: 0.9878790767013093
  UIQI: 0.9750621865713246
  ERGAS: 2.8410574992415762
  SAM: 2.194922924041748
  status: success
  error: 
  runtime_seconds: 253.419499238953
----------------------------------------------------------------------------------------------------
Sweep row 1
  scene_name: DC_data
  shape_policy: strict
  method_nam